# UC2_Returns_Fraud_Investigator

```
=============================================================================
UC2 — RETURNS FRAUD INVESTIGATOR
=============================================================================
Reads from : dbx_retail.gold.fact_returns
             dbx_retail.gold.dim_customers
             dbx_retail.gold.fact_orders
             dbx_retail.gold.fraud_rule_config  (runtime thresholds)
Writes to  : dbx_retail.gold.flagged_return_customers
             dbx_retail.gold.pipeline_run_log
Model      : databricks-gpt-oss-20b via Databricks AI Gateway
=============================================================================
FRAUD RULES (5 required, weights documented below)
-------------------------------------------------------
Rule 1 — high_return_volume         : 25 pts
  Customer has more than N total returns.
  Highest weight because raw volume is the primary fraud signal in retail.

Rule 2 — high_return_ratio          : 20 pts
  total_returns / total_orders > threshold.
  Returning more than half of everything ordered is abnormal at any volume.
  More defensible than raw counts alone.

Rule 3 — high_refund_value          : 20 pts
  Total refund value exceeds dollar threshold.
  Concentrated financial exposure — few customers, large payout liability.

Rule 4 — no_matching_order          : 25 pts
  Returns exist with order_id not present in fact_orders.
  Returning something you never ordered is definitionally suspicious.
  Joint-highest weight — this is the clearest data-backed fraud signal.

Rule 5 — cross_region_returns       : 10 pts
  Customer has returns recorded from more than one region.
  dbx_retail-specific rule: exploits unified view that was previously
  invisible across fragmented regional systems. Lower weight because
  legitimate customers can also exist across regions (e.g. relocated).

THRESHOLD : 35 points
  Triggering only Rule 2 alone (20 pts) = below threshold, no flag.
  Triggering Rule 1 + Rule 5 (35 pts)   = flagged for review.
  Triggering Rule 1 + Rule 4 (50 pts)   = high-priority investigation.
=============================================================================
```

In [0]:
%pip install openai
dbutils.library.restartPython()

In [0]:
import time
import json
from datetime import datetime, date
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType
)
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1"
)

MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC2_Returns_Fraud_Investigator"

print("LLM client ready.")
print(f"Run started at : {datetime.now().isoformat()}")

In [0]:

def extract_text(response) -> str:
    """
    databricks-gpt-oss-20b returns a structured list:
      [{"type": "reasoning", ...}, {"type": "text", "text": "..."}]
    Extract only the text block.
    """
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    """Calls LLM with exponential backoff. Returns error string on total failure."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_llm_output(text: str, min_words: int = 30) -> dict:
    """Validates LLM brief before writing to Delta."""
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable"]
    issues = []
    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(p in text.lower() for p in refusal_phrases):
        issues.append("refusal_detected")
    return {
        "text":         text,
        "quality_flag": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


print("Helper functions defined.")

In [0]:
# Thresholds are externalized so a data steward can adjust them
# without touching code. On first run, we seed the config table.

def seed_rule_config():
    """Creates the fraud_rule_config table with default thresholds."""
    config_data = [
        ("high_return_volume",   10.0,  True,  "Customer total returns exceeds this value"),
        ("high_return_ratio",     0.5,  True,  "Returns divided by orders exceeds this ratio"),
        ("high_refund_value",  1000.0,  True,  "Total refund value exceeds this dollar amount"),
        ("no_matching_order",     0.0,  True,  "Any unmatched return triggers this rule (binary)"),
        ("cross_region_returns",  1.0,  True,  "Returns from more than this many distinct regions"),
        ("flag_threshold",       35.0,  True,  "Customers scoring at or above this are flagged"),
    ]
    schema = StructType([
        StructField("rule_name",   StringType(), False),
        StructField("threshold",   FloatType(),  False),
        StructField("is_active",   StringType(), False),
        StructField("description", StringType(), True),
    ])
    # Convert bool to string for Delta compatibility
    rows = [(r[0], r[1], str(r[2]), r[3]) for r in config_data]
    (
        spark.createDataFrame(rows, schema=schema)
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("dbx_retail.gold.fraud_rule_config")
    )
    print("Seeded dbx_retail.gold.fraud_rule_config with default thresholds.")

try:
    config_df = spark.table("dbx_retail.gold.fraud_rule_config")
    if config_df.count() == 0:
        seed_rule_config()
        config_df = spark.table("dbx_retail.gold.fraud_rule_config")
except Exception:
    seed_rule_config()
    config_df = spark.table("dbx_retail.gold.fraud_rule_config")

# Load thresholds into a dict for use in scoring
thresholds = (
    config_df.filter(F.col("is_active") == "True")
    .select("rule_name", "threshold")
    .toPandas()
    .set_index("rule_name")["threshold"]
    .to_dict()
)

print("\nActive rule thresholds:")
for k, v in thresholds.items():
    print(f"  {k:<25} : {v}")

In [0]:

df_returns   = spark.table("dbx_retail.gold.fact_returns")
df_customers = spark.table("dbx_retail.gold.dim_customers")
df_orders    = spark.table("dbx_retail.gold.fact_orders")

print(f"\nfact_returns   : {df_returns.count()} rows")
print(f"dim_customers  : {df_customers.count()} rows")
print(f"fact_orders    : {df_orders.count()} rows")

display(df_returns.limit(5))

In [0]:

try:
    last_run_ts = spark.sql("""
        SELECT MAX(scored_at) FROM dbx_retail.gold.flagged_return_customers
    """).collect()[0][0]
    print(f"Last run: {last_run_ts}. Processing customers with new returns since then.")
except Exception:
    last_run_ts = None
    print("No previous run found. Scoring all customers.")

if last_run_ts is not None and "return_date" in df_returns.columns:
    new_customer_ids = (
        df_returns
        .filter(F.col("return_date") > F.lit(str(last_run_ts)))
        .select("customer_id")
        .distinct()
    )
    df_returns_inc = df_returns.join(new_customer_ids, on="customer_id", how="inner")
    print(f"Customers with new activity: {new_customer_ids.count()}")
else:
    df_returns_inc = df_returns
    print("Processing full returns table.")

In [0]:
# Aggregate per customer. DO NOT score individual returns — score the profile.
# This mirrors how the Service Anomaly notebook builds claim profiles.

df_order_valid_ids = df_orders.select("order_id").distinct()

# Find returns with no matching order — used for Rule 4
unmatched_returns = (
    df_returns_inc
    .join(df_order_valid_ids, on="order_id", how="left_anti")   # left_anti = not in orders
    .groupBy("customer_id")
    .agg(F.count("*").alias("unmatched_return_count"))
)

# Find cross-region return count per customer — used for Rule 5
cross_region = (
    df_returns_inc
    .join(df_orders.select("order_id", "region"), on="order_id", how="left")
    .groupBy("customer_id")
    .agg(F.countDistinct("region").alias("distinct_regions"))
)

# Core return profile
df_profiles = (
    df_returns_inc
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_returns"),
        F.sum("refund_amount").alias("total_refund_value"),
        F.avg("refund_amount").alias("avg_refund_value"),
        F.countDistinct("return_reason").alias("distinct_reasons"),
        F.collect_set("return_reason").alias("return_reasons_list")
    )
)

# Order count per customer for return ratio
df_order_counts = (
    df_orders
    .groupBy("customer_id")
    .agg(F.count("*").alias("total_orders"))
)

# Combine all profile components
df_profiles = (
    df_profiles
    .join(df_order_counts,   on="customer_id", how="left")
    .join(unmatched_returns, on="customer_id", how="left")
    .join(cross_region,      on="customer_id", how="left")
    .join(
        df_customers.select("customer_id", "customer_name", "segment", "region"),
        on="customer_id",
        how="left"
    )
    .fillna({
        "total_orders":          0,
        "unmatched_return_count": 0,
        "distinct_regions":       1
    })
    .withColumn(
        "return_ratio",
        F.when(F.col("total_orders") > 0,
               F.col("total_returns") / F.col("total_orders"))
        .otherwise(F.lit(1.0))   # no orders but has returns = 100% anomalous
    )
)

print(f"Customer profiles built: {df_profiles.count()}")
display(df_profiles.limit(5))

In [0]:
# Each rule adds a score column via withColumn.
# Thresholds are loaded from fraud_rule_config — not hardcoded.
# Pattern mirrors Service_Anomaly_Intelligence reference notebook.

t_volume   = float(thresholds.get("high_return_volume",   10))
t_ratio    = float(thresholds.get("high_return_ratio",     0.5))
t_value    = float(thresholds.get("high_refund_value",  1000))
t_regions  = float(thresholds.get("cross_region_returns",  1))
THRESHOLD  = int(thresholds.get("flag_threshold",         35))

df_scored = (
    df_profiles

    # Rule 1 — High return volume (25 pts)
    .withColumn("score_high_volume",
        F.when(F.col("total_returns") > t_volume, 25).otherwise(0))

    # Rule 2 — High return-to-order ratio (20 pts)
    .withColumn("score_high_ratio",
        F.when(F.col("return_ratio") > t_ratio, 20).otherwise(0))

    # Rule 3 — High total refund value (20 pts)
    .withColumn("score_high_refund_value",
        F.when(F.col("total_refund_value") > t_value, 20).otherwise(0))

    # Rule 4 — Returns with no matching order (25 pts)
    .withColumn("score_no_matching_order",
        F.when(F.col("unmatched_return_count") > 0, 25).otherwise(0))

    # Rule 5 — Cross-region return history (10 pts)
    .withColumn("score_cross_region",
        F.when(F.col("distinct_regions") > t_regions, 10).otherwise(0))

    # Total anomaly score
    .withColumn("total_score",
        F.col("score_high_volume")       +
        F.col("score_high_ratio")        +
        F.col("score_high_refund_value") +
        F.col("score_no_matching_order") +
        F.col("score_cross_region"))
)

print("Scoring complete.")
display(
    df_scored.select(
        "customer_id", "customer_name", "total_returns", "return_ratio",
        "total_refund_value", "unmatched_return_count", "distinct_regions",
        "total_score"
    ).orderBy(F.col("total_score").desc()).limit(10)
)

In [0]:

df_flagged = df_scored.filter(F.col("total_score") >= THRESHOLD)
flagged_count = df_flagged.count()

print(f"Flagged {flagged_count} customers for investigation (threshold >= {THRESHOLD})")
display(df_flagged.orderBy(F.col("total_score").desc()))

In [0]:
# The LLM explains — it does not detect.
# Detection is done by the rule engine above.
# Prompt is data-specific: it passes actual numbers, not generic descriptions.

def safe_get(row, field, default="N/A"):
    """Spark Row does not support .get() — use this helper."""
    try:
        val = row[field]
        return val if val is not None else default
    except Exception:
        return default


def build_brief_prompt(row) -> str:
    violated_rules = []
    if row["score_high_volume"]       > 0:
        violated_rules.append(
            f"  - High return volume       : {row['total_returns']} returns "
            f"(threshold > {t_volume}) | {row['score_high_volume']} pts")
    if row["score_high_ratio"]        > 0:
        violated_rules.append(
            f"  - High return-to-order ratio : {round(row['return_ratio']*100,1)}% "
            f"(threshold > {int(t_ratio*100)}%) | {row['score_high_ratio']} pts")
    if row["score_high_refund_value"] > 0:
        violated_rules.append(
            f"  - High total refund value  : ${round(safe_get(row,'total_refund_value',0),2)} "
            f"(threshold > ${t_value}) | {row['score_high_refund_value']} pts")
    if row["score_no_matching_order"] > 0:
        violated_rules.append(
            f"  - Returns without matching order : {row['unmatched_return_count']} records "
            f"| {row['score_no_matching_order']} pts")
    if row["score_cross_region"]      > 0:
        violated_rules.append(
            f"  - Cross-region activity    : returns from {row['distinct_regions']} regions "
            f"| {row['score_cross_region']} pts")

    reasons_text = ", ".join(safe_get(row, "return_reasons_list", []) or []) or "N/A"

    customer_section = (
        f"  Customer Name    : {safe_get(row, 'customer_name')}\n"
        f"  Segment          : {safe_get(row, 'segment')}\n"
        f"  Home Region      : {safe_get(row, 'region')}"
    )

    return f"""You are a fraud analyst at dbx_retail, a national retail chain.

A customer has been flagged by the automated anomaly scoring system with a risk score of
{row['total_score']} out of 100. The system is rule-based — it detects patterns,
you explain them.

CUSTOMER PROFILE:
{customer_section}
  Total Returns      : {row['total_returns']}
  Total Orders       : {safe_get(row, 'total_orders', 0)}
  Return-to-Order %  : {round(safe_get(row,'return_ratio',0)*100, 1)}%
  Total Refund Value : ${round(safe_get(row,'total_refund_value',0), 2)}
  Avg Refund Value   : ${round(safe_get(row,'avg_refund_value',0), 2)}
  Unmatched Returns  : {row['unmatched_return_count']}
  Distinct Regions   : {row['distinct_regions']}
  Return Reasons     : {reasons_text}

RULES VIOLATED:
{chr(10).join(violated_rules)}
  TOTAL RISK SCORE   : {row['total_score']} / 100

Write a 3-4 sentence investigation brief for the returns operations manager. Cover:
1. What specific patterns in this customer's data make this case suspicious — cite actual numbers
2. What innocent explanations might also explain these patterns
3. What the returns team should verify first — be specific and actionable

Plain English only. No bullet points. No markdown. No headers."""


SYSTEM_MSG_UC2 = (
    "You are a fraud analyst writing concise, data-grounded investigation briefs "
    "for a returns operations manager at dbx_retail retail. "
    "Write in plain English paragraphs only. No bullet points, no markdown, no headers."
)

print("Brief prompt builder ready.")

In [0]:

fraud_records  = []
llm_call_count = 0
review_count   = 0

for row in df_flagged.orderBy(F.col("total_score").desc()).collect():
    prompt    = build_brief_prompt(row)
    raw_brief = call_llm(prompt, SYSTEM_MSG_UC2)
    validated = validate_llm_output(raw_brief, min_words=30)
    llm_call_count += 1

    if validated["quality_flag"] == "REVIEW":
        review_count += 1

    rules_violated = "|".join([
        r for r, s in [
            ("high_return_volume",    row["score_high_volume"]),
            ("high_return_ratio",     row["score_high_ratio"]),
            ("high_refund_value",     row["score_high_refund_value"]),
            ("no_matching_order",     row["score_no_matching_order"]),
            ("cross_region_returns",  row["score_cross_region"])
        ] if s > 0
    ])

    fraud_records.append({
        "customer_id":            safe_get(row, "customer_id"),
        "customer_name":          safe_get(row, "customer_name"),
        "segment":                safe_get(row, "segment"),
        "region":                 safe_get(row, "region"),
        "total_returns":          int(safe_get(row, "total_returns", 0)),
        "total_orders":           int(safe_get(row, "total_orders",  0)),
        "return_ratio":           float(round(safe_get(row, "return_ratio", 0), 4)),
        "total_refund_value":     float(round(safe_get(row, "total_refund_value", 0), 2)),
        "unmatched_return_count": int(safe_get(row, "unmatched_return_count", 0)),
        "distinct_regions":       int(safe_get(row, "distinct_regions", 1)),
        "anomaly_score":          int(row["total_score"]),
        "rules_violated":         rules_violated,
        "investigation_brief":    validated["text"],
        "quality_flag":           validated["quality_flag"],
        "quality_issues":         validated["issues"],
        "scored_at":              datetime.now().isoformat()
    })

    # Print output in the same style as Service_Anomaly_Intelligence reference
    print("=" * 70)
    print(f"CUSTOMER ID     : {safe_get(row, 'customer_id')}")
    print(f"CUSTOMER NAME   : {safe_get(row, 'customer_name')}")
    print(f"RISK SCORE      : {row['total_score']} / 100")
    print(f"\nRULES VIOLATED:")
    if row["score_high_volume"]       > 0: print(f"  [x] High return volume       : {row['score_high_volume']} pts")
    if row["score_high_ratio"]        > 0: print(f"  [x] High return-to-order ratio : {row['score_high_ratio']} pts")
    if row["score_high_refund_value"] > 0: print(f"  [x] High refund value        : {row['score_high_refund_value']} pts")
    if row["score_no_matching_order"] > 0: print(f"  [x] No matching order        : {row['score_no_matching_order']} pts")
    if row["score_cross_region"]      > 0: print(f"  [x] Cross-region activity    : {row['score_cross_region']} pts")
    print(f"\nQUALITY FLAG    : {validated['quality_flag']}")
    print(f"\nINVESTIGATION BRIEF:")
    print(validated["text"])
    print("=" * 70 + "\n")

print(f"\nTotal flagged: {len(fraud_records)} | LLM calls: {llm_call_count} | For review: {review_count}")

In [0]:

schema = StructType([
    StructField("customer_id",            StringType(),  True),
    StructField("customer_name",          StringType(),  True),
    StructField("segment",                StringType(),  True),
    StructField("region",                 StringType(),  True),
    StructField("total_returns",          IntegerType(), True),
    StructField("total_orders",           IntegerType(), True),
    StructField("return_ratio",           FloatType(),   True),
    StructField("total_refund_value",     FloatType(),   True),
    StructField("unmatched_return_count", IntegerType(), True),
    StructField("distinct_regions",       IntegerType(), True),
    StructField("anomaly_score",          IntegerType(), True),
    StructField("rules_violated",         StringType(),  True),
    StructField("investigation_brief",    StringType(),  True),
    StructField("quality_flag",           StringType(),  True),
    StructField("quality_issues",         StringType(),  True),
    StructField("scored_at",              StringType(),  True)
])

df_fraud_output = spark.createDataFrame(fraud_records, schema=schema)

(
    df_fraud_output.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.flagged_return_customers")
)

print("Written to dbx_retail.gold.flagged_return_customers")
display(
    spark.table("dbx_retail.gold.flagged_return_customers")
    .orderBy(F.col("anomaly_score").desc())
)

In [0]:

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": df_profiles.count(),
    "records_flagged":   flagged_count,
    "llm_calls_made":    llm_call_count,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS",
    "notes":             f"threshold={THRESHOLD} | watermark={last_run_ts is not None}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.pipeline_run_log")
)

print(f"Run log written. Status: {run_log[0]['status']}")